# Week 2, Day 2 — SQL

**CSC 381/576 · Sep 2, 2026**

Same data as last week. This time it lives in a database and we ask it questions
in a different language.

Two tables:

| Table | What one row is |
|---|---|
| `listings` | One used car for sale — last week's cleaned file, plus a `dealer_id` |
| `dealers` | One dealership |

The dealer's name, city, and year opened are **not** in `listings`. That is not an
oversight — it is how databases are built. You do not write "Brandywine Auto
Group, Downingtown, PA, 1987" onto twelve rows; you write `D-04` twelve times and
keep one row about D-04 somewhere else. Getting the two back together is a `JOIN`,
and it is the last third of tonight.

## 0 · Setup — the six lines from Monday

Nothing to install. `sqlite3` is in the Python standard library.

In [1]:
import pandas as pd
import sqlite3

listings = pd.read_csv('data/listings.csv')
dealers  = pd.read_csv('data/dealers.csv')

con = sqlite3.connect(':memory:')
listings.to_sql('listings', con, index=False, if_exists='replace')
dealers.to_sql('dealers',   con, index=False, if_exists='replace')

def q(sql):
    """Run a query, get a DataFrame back."""
    return pd.read_sql(sql, con)

print(f'listings: {len(listings)} rows   dealers: {len(dealers)} rows')

listings: 42 rows   dealers: 6 rows


`q()` is doing the whole trick: SQL goes in as a string, and what comes back is a
DataFrame you already know how to use. **SQL does not replace pandas. It is how
you get the data into pandas in the first place.**

## 1 · SELECT and FROM — the two words you cannot avoid

`SELECT` picks columns. `FROM` says which table.

In [2]:
q("""
    SELECT *
    FROM   listings
    LIMIT  5
""")

,listing_id,dealer_id,make,model,year,price,mileage,color,transmission,listed_on,seller,flag_any
0,L-1001,D-01,Toyota,Camry,2015,12500.0,78000.0,Silver,Automatic,2026-08-01,CarMax,0
1,L-1002,NaN,Toyota,Corolla,2018,14750.0,52300.0,Unknown,Automatic,2026-08-03,Private,0
2,L-1003,NaN,Honda,Civic,2105,9800.0,96000.0,Blue,Manual,2026-08-05,Private,1
3,L-1004,D-02,Toyota,RAV4,2019,21000.0,25787.0,Red,Automatic,2026-08-05,Dealer,0
4,L-1005,D-01,Toyota,Camry,2016,11900.0,88000.0,Unknown,Automatic,2026-08-07,CarMax,0


`SELECT *` is `df.head()` with extra steps. It is fine while you are looking
around and a bad habit in anything you save — name the columns you want, so that
your query keeps working when somebody adds a column next March.

In [3]:
q("""
    SELECT listing_id, make, model, year, price
    FROM   listings
    LIMIT  5
""")

,listing_id,make,model,year,price
0,L-1001,Toyota,Camry,2015,12500.0
1,L-1002,Toyota,Corolla,2018,14750.0
2,L-1003,Honda,Civic,2105,9800.0
3,L-1004,Toyota,RAV4,2019,21000.0
4,L-1005,Toyota,Camry,2016,11900.0


## 2 · WHERE — pick rows

`SELECT` picks columns, `WHERE` picks rows. That is the whole division of labour.

In pandas this was `df[df['make'] == 'Toyota']`.

In [4]:
q("""
    SELECT listing_id, make, model, year, price
    FROM   listings
    WHERE  make = 'Toyota'
""")

,listing_id,make,model,year,price
0,L-1001,Toyota,Camry,2015,12500.0
1,L-1002,Toyota,Corolla,2018,14750.0
2,L-1004,Toyota,RAV4,2019,21000.0
3,L-1005,Toyota,Camry,2016,11900.0
4,L-1013,Toyota,Highlander,2015,18900.0
5,L-1016,Toyota,Camry,2017,14200.0
6,L-1020,Toyota,Tacoma,2019,32400.0
7,L-1024,Toyota,Corolla,2020,18600.0
8,L-1028,Toyota,Prius,2018,19300.0
9,L-1032,Toyota,4Runner,2019,34100.0


Conditions combine with `AND` / `OR`, and compare with `= < > <= >= !=`.

Two that trip everyone up on day one:

- **Strings use single quotes.** `'Toyota'`, not `"Toyota"`.
- **Equality is one `=`**, not `==`. SQL is not Python.

In [5]:
q("""
    SELECT listing_id, make, model, year, price, mileage
    FROM   listings
    WHERE  make = 'Toyota'
      AND  mileage < 80000
      AND  year >= 2016
""")

,listing_id,make,model,year,price,mileage
0,L-1002,Toyota,Corolla,2018,14750.0,52300.0
1,L-1004,Toyota,RAV4,2019,21000.0,25787.0
2,L-1016,Toyota,Camry,2017,14200.0,72000.0
3,L-1020,Toyota,Tacoma,2019,32400.0,45600.0
4,L-1024,Toyota,Corolla,2020,18600.0,31200.0
5,L-1028,Toyota,Prius,2018,19300.0,41943.0
6,L-1032,Toyota,4Runner,2019,34100.0,39700.0


### The one you will actually need: `flag_any`

Last week we flagged the seven impossible rows instead of deleting them —
`price = 0`, `year = 2105`, negative mileage. The flag is still in the table, and
now you can see why keeping it was worth the trouble: **whoever uses this data
decides what to do about those rows, and they decide it in one line.**

In [6]:
q("""
    SELECT listing_id, make, year, price, mileage
    FROM   listings
    WHERE  flag_any = 1
""")

,listing_id,make,year,price,mileage
0,L-1003,Honda,2105,9800.0,96000.0
1,L-1006,Ford,2017,1.0,120000.0
2,L-1007,Honda,2017,16200.0,-4500.0
3,L-1009,Chevrolet,2014,0.0,142000.0
4,L-1025,Ford,2015,2.0,149000.0
5,L-1030,Ford,1015,27800.0,54000.0
6,L-1033,Honda,2016,13850.0,-1200.0


Note `= 1`, not `= True`. SQLite has no boolean type — it stores our `True`/`False`
as `1`/`0`. This is exactly the "one column, two ideas about what a value is"
problem from last week, except now it is the *database* doing it to you. Every
database has a handful of these; you learn them per database.

## 3 · ORDER BY — sort

`DESC` for descending, `ASC` (the default) for ascending. `LIMIT` cuts it short.

In [7]:
q("""
    SELECT   listing_id, make, model, year, price
    FROM     listings
    WHERE    flag_any = 0
    ORDER BY price DESC
    LIMIT    5
""")

,listing_id,make,model,year,price
0,L-1032,Toyota,4Runner,2019,34100.0
1,L-1020,Toyota,Tacoma,2019,32400.0
2,L-1035,Ford,Ranger,2020,26300.0
3,L-1040,Ford,Mustang,2018,25400.0
4,L-1023,Chevrolet,Silverado,2016,24900.0


`ORDER BY ... LIMIT` is "top N", and it is probably the single most-run query
shape in the working world.

Notice `WHERE flag_any = 0` doing real work here: without it, the cheapest-car
query returns the $0 and $1 rows and you would have concluded something false
about the market.

## 4 · GROUP BY — one row per group

This is the one that matters. `GROUP BY make` says: **collapse all the Toyota rows
into one row.** Once you have done that, every column you select has to be either
the thing you grouped by, or a summary of the rows inside the group.

Those summaries are the aggregate functions: `COUNT`, `AVG`, `SUM`, `MIN`, `MAX`.

In [8]:
q("""
    SELECT   make,
             COUNT(*)   AS n,
             AVG(price) AS avg_price,
             MIN(price) AS cheapest,
             MAX(price) AS dearest
    FROM     listings
    WHERE    flag_any = 0
    GROUP BY make
    ORDER BY n DESC
""")

,make,n,avg_price,cheapest,dearest
0,Toyota,11,19004.545455,11400.0,34100.0
1,Honda,7,18935.714286,14100.0,23100.0
2,Nissan,5,16600.000000,10750.0,21800.0
3,Ford,5,16960.000000,6400.0,26300.0
4,Chevrolet,4,16850.000000,8900.0,24900.0
5,Hyundai,3,16466.666667,13300.0,20900.0


`AS` renames a column. Without it your column is called `AVG(price)`, which is
legal and horrible to work with afterwards. **Name every computed column.**

### Stop and look at what you just wrote

Count, mean, min, max, grouped by a category. **That is descriptive statistics.**
You have been doing statistics for the last thirty seconds and calling it SQL.

Same numbers, the pandas way:

In [9]:
clean = listings[listings['flag_any'] == False]

(clean.groupby('make')['price']
      .agg(n='count', avg_price='mean', cheapest='min', dearest='max')
      .sort_values('n', ascending=False))

,n,avg_price,cheapest,dearest
make,,,,
Toyota,11,19004.545455,11400.0,34100.0
Honda,7,18935.714286,14100.0,23100.0
Nissan,5,16600.000000,10750.0,21800.0
Ford,5,16960.000000,6400.0,26300.0
Chevrolet,4,16850.000000,8900.0,24900.0
Hyundai,3,16466.666667,13300.0,20900.0


| The question | SQL | pandas |
|---|---|---|
| Which columns? | `SELECT` | `df[[...]]` |
| Which rows? | `WHERE` | boolean indexing |
| One row per group | `GROUP BY` | `.groupby()` |
| Summarise the group | `AVG`, `COUNT`, `MIN`, `MAX` | `.agg('mean', 'count', ...)` |
| Sort | `ORDER BY` | `.sort_values()` |
| First N | `LIMIT` | `.head()` |

Six ideas. Both languages. **Week 3 takes these same numbers and draws them.**

### `HAVING` — the filter that runs *after* grouping

`WHERE` filters rows before they are grouped. `HAVING` filters the groups
themselves. "Only makes with at least five cars" is a `HAVING`.

In [10]:
q("""
    SELECT   make, COUNT(*) AS n, AVG(price) AS avg_price
    FROM     listings
    WHERE    flag_any = 0
    GROUP BY make
    HAVING   COUNT(*) >= 5
    ORDER BY avg_price DESC
""")

,make,n,avg_price
0,Toyota,11,19004.545455
1,Honda,7,18935.714286
2,Ford,5,16960.000000
3,Nissan,5,16600.000000


---

# Part 2 · JOIN

Everything above used one table. Now the interesting part.

In [11]:
q("SELECT * FROM dealers")

,dealer_id,dealer_name,city,state,opened
0,D-01,CarMax,Devon,PA,2004
1,D-02,Chadds Ford Motors,Chadds Ford,PA,1998
2,D-03,Gay Street Auto,West Chester,PA,2011
3,D-04,Brandywine Auto Group,Downingtown,PA,1987
4,D-05,Main Line Used Cars,Wayne,PA,2015
5,D-06,Route 30 Motors,Exton,PA,2021


Six dealers. `listings` has a `dealer_id` column that points at this table.
To put a dealer's **name** next to a car, you have to join.

In [12]:
q("""
    SELECT   l.listing_id, l.make, l.model, l.price,
             d.dealer_name, d.city
    FROM     listings l
    JOIN     dealers  d  ON l.dealer_id = d.dealer_id
    ORDER BY d.dealer_name
    LIMIT    10
""")

,listing_id,make,model,price,dealer_name,city
0,L-1010,Honda,Civic,17300.0,Brandywine Auto Group,Downingtown
1,L-1020,Toyota,Tacoma,32400.0,Brandywine Auto Group,Downingtown
2,L-1032,Toyota,4Runner,34100.0,Brandywine Auto Group,Downingtown
3,L-1001,Toyota,Camry,12500.0,CarMax,Devon
4,L-1005,Toyota,Camry,11900.0,CarMax,Devon
5,L-1013,Toyota,Highlander,18900.0,CarMax,Devon
6,L-1017,Honda,Odyssey,15750.0,CarMax,Devon
7,L-1022,Honda,Pilot,20100.0,CarMax,Devon
8,L-1028,Toyota,Prius,19300.0,CarMax,Devon
9,L-1034,Nissan,Murano,18700.0,CarMax,Devon


Three new things in that query:

- **`l` and `d` are aliases.** `FROM listings l` means "call it `l` from here on".
  Once two tables are in play, `dealer_id` is ambiguous — you have to say which
  table's `dealer_id` you mean.
- **`ON` is the matching rule.** "A listings row and a dealers row belong together
  when their `dealer_id` values are equal."
- **`JOIN` on its own means `INNER JOIN`.** Which is where the problem starts.

## The trap

Run the same join, but count the rows instead of looking at them.

In [13]:
q("""
    SELECT COUNT(*) AS rows_after_join
    FROM   listings l
    JOIN   dealers  d ON l.dealer_id = d.dealer_id
""")

,rows_after_join
0,22


**22.** We started with 42 listings and the join returned 22.

Nothing errored. Nothing warned you. Twenty rows — nearly half the table — were
silently dropped, and if you had gone straight to `AVG(price)` you would have
published an average price for the used-car market that quietly excludes every
private seller in it.

Why: an `INNER JOIN` keeps only rows that found a partner. Private-party and
auction listings have no dealer, so their `dealer_id` is empty, so they match
nothing, so they vanish.

In [14]:
q("""
    SELECT   seller, COUNT(*) AS n
    FROM     listings
    WHERE    dealer_id IS NULL OR dealer_id = ''
    GROUP BY seller
""")

,seller,n
0,Auction,4
1,Private,16


`IS NULL`, not `= NULL`. Nothing is ever *equal* to NULL in SQL, including NULL —
because NULL means "unknown", and two unknowns are not known to be the same.
This is the single weirdest rule in the language and it catches everybody once.

## `LEFT JOIN` — keep everything on the left

A `LEFT JOIN` keeps every row of the left-hand table whether or not it found a
partner, and fills the missing side with NULL.

In [15]:
q("""
    SELECT COUNT(*) AS rows_after_left_join
    FROM   listings l
    LEFT JOIN dealers d ON l.dealer_id = d.dealer_id
""")

,rows_after_left_join
0,42


In [16]:
q("""
    SELECT   l.listing_id, l.seller, l.price, d.dealer_name
    FROM     listings l
    LEFT JOIN dealers d ON l.dealer_id = d.dealer_id
    WHERE    d.dealer_name IS NULL
    LIMIT    6
""")

,listing_id,seller,price,dealer_name
0,L-1002,Private,14750.0,None
1,L-1003,Private,9800.0,None
2,L-1006,Auction,1.0,None
3,L-1007,Private,16200.0,None
4,L-1009,Private,0.0,None
5,L-1011,Private,10750.0,None


42 rows, and the private sellers are still there with `None` for a dealer name —
which is the truth. They really do not have a dealer.

**The rule to leave with: after every join, check the row count.** Same habit as
last week, when we printed the count after every step that could drop rows. It is
the same failure, in a different language.

## It goes wrong from the other side too

In [17]:
q("""
    SELECT   d.dealer_name, d.city, COUNT(l.listing_id) AS n_cars
    FROM     dealers d
    LEFT JOIN listings l ON d.dealer_id = l.dealer_id
    GROUP BY d.dealer_id, d.dealer_name, d.city
    ORDER BY n_cars DESC
""")

,dealer_name,city,n_cars
0,CarMax,Devon,8
1,Chadds Ford Motors,Chadds Ford,4
2,Gay Street Auto,West Chester,4
3,Brandywine Auto Group,Downingtown,3
4,Main Line Used Cars,Wayne,3
5,Route 30 Motors,Exton,0


Route 30 Motors has zero cars listed. With an `INNER JOIN` it disappears from the
report entirely — and "the dealer with no inventory" is very often exactly the row
somebody needed to see.

`COUNT(l.listing_id)` rather than `COUNT(*)` is deliberate: `COUNT(*)` counts
rows, and the LEFT JOIN gives Route 30 one row full of NULLs, so `COUNT(*)` would
say 1. `COUNT(column)` skips NULLs and correctly says 0.

## Putting it together

Everything from tonight, in one query: join, filter, group, aggregate, filter the
groups, sort.

In [18]:
q("""
    SELECT   d.dealer_name,
             d.city,
             COUNT(*)              AS n_cars,
             ROUND(AVG(l.price))   AS avg_price,
             ROUND(AVG(l.mileage)) AS avg_miles
    FROM     listings l
    JOIN     dealers  d ON l.dealer_id = d.dealer_id
    WHERE    l.flag_any = 0
    GROUP BY d.dealer_id, d.dealer_name, d.city
    HAVING   COUNT(*) >= 3
    ORDER BY avg_price DESC
""")

,dealer_name,city,n_cars,avg_price,avg_miles
0,Brandywine Auto Group,Downingtown,3,27933.0,43100.0
1,Main Line Used Cars,Wayne,3,22667.0,32333.0
2,Gay Street Auto,West Chester,3,22217.0,34333.0
3,Chadds Ford Motors,Chadds Ford,4,20075.0,32198.0
4,CarMax,Devon,8,17119.0,74855.0


An `INNER JOIN` is right *here*, because the question is about dealers. Every
join is a decision, and like every cleaning decision last week, it is one you
should be able to say a sentence about.

**Read the clauses in this order** — it is the order the database runs them, and
it is not the order you write them:

```
FROM / JOIN  ->  WHERE  ->  GROUP BY  ->  HAVING  ->  SELECT  ->  ORDER BY
```

That order explains `HAVING`: by the time it runs, the groups exist. It also
explains why you cannot use a `SELECT` alias inside `WHERE` — `WHERE` ran first.

---

# Part 3 · The same query somewhere else

You will not run this tonight and it is not on any homework. Look at it for
thirty seconds.

Apache Spark runs a query across a cluster of machines instead of one. Here is
tonight's grouping, in Spark:

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('cars').getOrCreate()
df = spark.read.csv('listings.csv', header=True, inferSchema=True)
df.createOrReplaceTempView('listings')

spark.sql("""
    SELECT   make, COUNT(*) AS n, AVG(price) AS avg_price
    FROM     listings
    WHERE    flag_any = 0
    GROUP BY make
""").show()
```

The query in the middle is **character-for-character the query you wrote an hour
ago**. That is the point. The thing you learned tonight is not "SQLite" — it is a
language that sits on top of a file, a company database, a warehouse holding
petabytes, and a cluster of five hundred machines, essentially unchanged.

What changes is what happens underneath: SQLite runs it in this notebook's
memory, Spark splits it across machines and reassembles the answer. You reach for
Spark when the data genuinely does not fit on one computer. Ours is 42 rows, so
Spark would be slower than pandas, and roughly a thousand times more annoying.

---

## Before Monday

- **HW1 Part 4** is now unblocked — it is two queries, and they are both shapes
  you wrote tonight.
- Optional: `sqlbolt.com` lessons 1–6, about 45 minutes in your browser.

**Monday, Sep 8 — no class.** Labor Day. Next class is **Wednesday, Sep 9**, and
we start drawing these numbers instead of printing them.